# NoClip · Difix 실험 (Google Colab)

구간 하나를 **① 기본 학습 → ② Difix 보정 학습**으로 돌려서, 벽 쪽(옆) 각도의 줄무늬·비침이 줄어드는지 비교함 (해결 방향 4).
② 는 학습 중간중간 카메라를 벽 쪽으로 돌린 각도에서 렌더링 → NVIDIA Difix 로 깨진 부분을 고침 → 그 사진을 학습 데이터에 추가하는 방식.

### 준비 (PC 에서 한 번)
`F:\Sample\6\1b\data` 처럼 **`images` 와 `sparse` 가 들어 있는 폴더**를 구글 드라이브에 올려 둠.
드라이브 안 위치는 상관없음 → 1번 셀의 `DRIVE_DATA` 를 그 폴더로 지정. (`gs2d_colab` 과 같은 폴더를 써도 됨)

### 실행
1. **런타임 → 런타임 유형 변경 → T4 GPU** (L4 가 보이면 L4)
2. 1번 셀에서 `DRIVE_DATA`, `SCENE` 확인
3. **런타임 → 모두 실행**. 문제가 생기면 그 셀에서 멈추고 이유를 출력함

### 걸리는 시간 (T4, 복도 1 기준 추정)
설치 10~30분 + ① 약 1시간 + ② 약 3시간. 무료 GPU 한도 안에 다 안 끝날 수 있음 →
**끊기면 다시 "모두 실행" 하면 마지막 저장 지점(5,000 스텝마다)부터 이어서 함**. 끝난 단계는 건너뜀.

### 바꾼 점 (방 6 실험 이후)
- 기본 학습 방식을 gsplat 기본(default) → **MCMC + 스플랫 300만 개 상한**으로 변경. 방 6 에서 기본 방식은 floater·덕지덕지한 질감이 많았음. Brush 와 같은 상한이라 비교도 공정함
- 보정 라운드 8번(4,000 스텝마다), 라운드당 약 120장 → 구간 크기와 상관없이 시간이 일정함
- 4K 원본을 로컬에 통째로 복사하지 않고 줄인 사진만 만듦 (빠르고 디스크 절약)

결과(체크포인트, PLY, 비교 이미지)는 드라이브 `3DGS/3_학습결과/4차_colab/out/<SCENE>` 에 저장. 끝나면 **런타임 → 런타임 연결 해제 및 삭제**.
라이선스: Difix 는 NVIDIA 비상업 라이선스.

## 1. GPU 확인 · 드라이브 연결 · 설정

In [ ]:
import os, subprocess
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    raise RuntimeError("GPU 런타임이 아님 -> 런타임 > 런타임 유형 변경 에서 T4/L4 선택. "
                       "선택지가 없으면 무료 GPU 한도 소진 상태이니 나중에 다시 시도")

from google.colab import drive
drive.mount('/content/drive')

# ── 여기만 바꾸면 됨 ──────────────────────────────────────────────
# images/ 와 sparse/ 가 들어 있는 드라이브 폴더 (예: 3DGS/2_정합데이터/4차/1 = 4차 촬영 구간 1)
DRIVE_DATA = "/content/drive/MyDrive/3DGS/2_정합데이터/4차/1"
SCENE      = "corridor1_fill"   # 결과 폴더 이름
# 빈 곳 채우기 (tools/fill 로 만든 보강 시작점). 쓰지 않으려면 None
FILL_POINTS = "/content/drive/MyDrive/3DGS/2_정합데이터/4차/1_fill/points3D.bin"
PITCH_MAX  = 40              # 위(천장)/아래(바닥)로 돌려 보정할 최대 각도. 0 이면 좌우만
NO_ALPHA   = True            # 렌더링이 비어 있던 곳(천장 등)도 보정 사진으로 학습
FACTOR     = 4               # 4K -> 960x540
SH         = 2               # Brush 와 동일
CAP        = 3_000_000       # 스플랫 개수 상한 (Brush 와 동일)
RUN_RBG    = False           # True 면 ③ 배경 무작위화 실험도 돌림 (+1시간)
# ─────────────────────────────────────────────────────────────────

DATA = f"/content/data/{SCENE}"                                   # 로컬 작업 폴더
OUT  = f"/content/drive/MyDrive/3DGS/3_학습결과/4차_colab/out/{SCENE}"
GS   = "/content/Difix3D/examples/gsplat"
os.makedirs(OUT, exist_ok=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

for sub in ["images", "sparse/0"]:
    if not os.path.isdir(f"{DRIVE_DATA}/{sub}"):
        raise RuntimeError(f"{DRIVE_DATA}/{sub} 가 없음. DRIVE_DATA 경로나 드라이브 업로드(동기화)를 확인. "
                           f"현재 내용: {os.listdir(DRIVE_DATA) if os.path.isdir(DRIVE_DATA) else '폴더 없음'}")
N_IMG = len(os.listdir(f"{DRIVE_DATA}/images"))
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip())
print("사진", N_IMG, "장 /", DRIVE_DATA)
print("결과 ->", OUT)

## 2. 설치 (10~20분)
런타임이 새로 켜질 때마다 다시 필요함. 이미 설치돼 있으면 건너뜀. 세션 다시 시작은 필요 없음.

In [ ]:
import os, subprocess, glob

def ok():
    return subprocess.run(["python", "-c", "import gsplat, pycolmap, diffusers, fused_ssim"],
                          capture_output=True).returncode == 0 and os.path.exists(GS)

if ok():
    print("이미 설치됨 (건너뜀)")
else:
    if not os.path.exists(GS):
        !git clone -q https://github.com/nv-tlabs/Difix3D.git /content/Difix3D
    !pip install -q ninja
    !pip install -q diffusers==0.25.1 transformers==4.38.0 huggingface-hub==0.25.1 peft==0.9.0 \
        einops lpips "imageio[ffmpeg]" viser nerfview==0.0.2 "tyro>=0.8.8" tensorly scikit-learn \
        "torchmetrics[image]" tensorboard opencv-python pyyaml
    !pip install -q git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e
    !pip install -q --no-build-isolation git+https://github.com/rahul-goel/fused-ssim@1272e21a282342e89537159e4bad508b19b34157
    !pip install -q gsplat

# Colab 에 깔린 HuggingFace 'datasets' 패키지와 이름이 겹침 -> 로컬 폴더를 정식 패키지로 만들어 우선시킴
open(f"{GS}/datasets/__init__.py", "a").close()
# pycolmap 을 numpy 2 에서 쓸 수 있게 한 줄 수정 (np.uint64(-1) -> 같은 값)
for f in glob.glob("/usr/local/lib/python3*/dist-packages/pycolmap/scene_manager.py"):
    s = open(f).read()
    if "np.uint64(-1)" in s:
        open(f, "w").write(s.replace("np.uint64(-1)", "np.uint64(np.iinfo(np.uint64).max)")); print("patched", f)

if not ok():
    raise RuntimeError("설치가 덜 됨 - 위 출력의 ERROR 부분을 확인")
print("설치 OK")

## 3. 보조 스크립트 · 학습 코드 수정
보정할 각도 = 학습 카메라를 벽 쪽(좌우)으로 돌린 각도. 라운드마다 15° 씩 더 돌려서 최대 60°.

In [ ]:
import os; os.makedirs('/content/nc', exist_ok=True)

In [ ]:
%%writefile /content/nc/nc_patch.py
"""Difix3D gsplat 트레이너를 우리 촬영 데이터(복도/방)에 맞게 수정.
1) Difix 모델은 처음 보정할 때만 불러옴 (기본 학습 중 GPU 메모리 절약)
2) 보정할 '새 각도' = 학습 카메라를 좌우로 돌려 벽을 보게 한 각도.
   라운드마다 NC_STEP_DEG 씩 더 돌리고, 최대 NC_MAX_DEG 까지.
   NC_PITCH_MAX > 0 이면 위(천장)/아래(바닥)로 돌린 각도도 추가 (NC_PITCH_STEP 씩, 최대 NC_PITCH_MAX).
   NC_NO_ALPHA=1 이면 렌더링이 비어 있던 곳도 보정 사진으로 학습 (빈 곳 채우기).
"""
import sys

path = sys.argv[1]
src = open(path, encoding="utf-8").read()
if "[NoClip]" in src:
    print("이미 수정됨"); sys.exit(0)

old_init = '''        # Diffusion fixer
        self.difix = DifixPipeline.from_pretrained("nvidia/difix_ref", trust_remote_code=True)
        self.difix.set_progress_bar_config(disable=True)
        self.difix.to("cuda")
'''
new_init = '''        # Diffusion fixer (loaded lazily on first fix)
        self.difix = None
'''
old_fix = '''        if len(self.cfg.fix_steps) == 1:
            novel_poses = self.parser.camtoworlds[self.valset.indices]
        else:
            novel_poses = self.interpolator.shift_poses(self.current_novel_poses, self.parser.camtoworlds[self.valset.indices], distance=0.5)
'''
new_fix = '''        if self.difix is None:
            self.difix = DifixPipeline.from_pretrained("nvidia/difix_ref", trust_remote_code=True)
            self.difix.set_progress_bar_config(disable=True)
            self.difix.to("cuda")
        # [NoClip] look sideways toward the walls, a little more every round
        k = len(self.novelloaders) + 1
        max_deg = float(os.environ.get("NC_MAX_DEG", 60))
        step_deg = float(os.environ.get("NC_STEP_DEG", 15))
        stride = int(os.environ.get("NC_STRIDE", 8))
        ang = np.deg2rad(min(max_deg, k * step_deg))
        base = self.parser.camtoworlds[np.array(self.trainset.indices)[::stride]]
        rotated = []
        for sgn in (1.0, -1.0):
            c, s = np.cos(sgn * ang), np.sin(sgn * ang)
            Ry = np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])
            p = base.copy()
            p[:, :3, :3] = base[:, :3, :3] @ Ry  # yaw around the camera's own up/down axis
            rotated.append(p)
        # [NoClip] also look up (ceiling) / down (floor) when NC_PITCH_MAX > 0
        pmax = float(os.environ.get("NC_PITCH_MAX", 0))
        if pmax > 0:
            pang = np.deg2rad(min(pmax, k * float(os.environ.get("NC_PITCH_STEP", 10))))
            for sgn in (1.0, -1.0):
                c, s = np.cos(sgn * pang), np.sin(sgn * pang)
                Rx = np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]])
                p = base.copy()
                p[:, :3, :3] = base[:, :3, :3] @ Rx  # pitch around the camera's own right axis
                rotated.append(p)
        novel_poses = np.concatenate(rotated, 0)
        print(f"[NoClip] fix round {k}: {len(novel_poses)} views, yaw +-{np.rad2deg(ang):.0f} deg"
              + (f", pitch +-{np.rad2deg(pang):.0f} deg" if pmax > 0 else ""))
'''
old_alpha = '''        parser.alpha_mask_paths = [f"{self.render_dir}/novel/{step}/Alpha/{i:04d}.png" for i in range(len(novel_poses))]
'''
new_alpha = '''        parser.alpha_mask_paths = [f"{self.render_dir}/novel/{step}/Alpha/{i:04d}.png" for i in range(len(novel_poses))]
        if os.environ.get("NC_NO_ALPHA") == "1":  # [NoClip] also learn where the render was empty
            parser.alpha_mask_paths = None
'''
for o, n in [(old_init, new_init), (old_fix, new_fix), (old_alpha, new_alpha)]:
    assert src.count(o) == 1, "패치 위치를 찾지 못함: " + o[:60]
    src = src.replace(o, n)
open(path, "w", encoding="utf-8").write(src)
print("수정 완료:", path)

In [ ]:
%%writefile /content/nc/nc_export_ply.py
"""gsplat 체크포인트(.pt) -> 표준 3DGS PLY (Unity / SuperSplat 용)
사용: python nc_export_ply.py ckpt_59999_rank0.pt out.ply
"""
import sys
import numpy as np
import torch

ckpt, out = sys.argv[1], sys.argv[2]
s = torch.load(ckpt, map_location="cpu", weights_only=True)["splats"]
means = s["means"].float().numpy()
scales = s["scales"].float().numpy()                       # log scale
quats = torch.nn.functional.normalize(s["quats"].float(), dim=-1).numpy()  # w x y z
opac = s["opacities"].float().numpy()[:, None]             # logit
sh0 = s["sh0"].float().numpy().reshape(len(means), 3)
shN = s["shN"].float().numpy()                              # [N, K, 3]
f_rest = shN.transpose(0, 2, 1).reshape(len(means), -1)    # channel-major like original 3DGS
normals = np.zeros_like(means)

names = ["x", "y", "z", "nx", "ny", "nz"]
names += [f"f_dc_{i}" for i in range(3)]
names += [f"f_rest_{i}" for i in range(f_rest.shape[1])]
names += ["opacity"] + [f"scale_{i}" for i in range(3)] + [f"rot_{i}" for i in range(4)]
data = np.concatenate([means, normals, sh0, f_rest, opac, scales, quats], 1).astype(np.float32)
keep = np.isfinite(data).all(1)
data = data[keep]

with open(out, "wb") as f:
    head = "ply\nformat binary_little_endian 1.0\n"
    head += f"element vertex {len(data)}\n"
    head += "".join(f"property float {n}\n" for n in names)
    head += "end_header\n"
    f.write(head.encode("ascii"))
    f.write(data.tobytes())
print(f"{out}: {len(data):,} splats (SH degree {int(np.sqrt(shN.shape[1] + 1)) - 1})")

In [ ]:
%%writefile /content/nc/nc_compare.py
"""같은 위치에서 카메라를 옆으로 돌려(벽 쪽) 여러 결과를 나란히 렌더링 -> PNG
사용 (examples/gsplat 폴더에서):
  python nc_compare.py DATA_DIR FACTOR OUT.png 이름1=ckpt1.pt 이름2=ckpt2.pt ...
"""
import sys
import numpy as np
import torch
from PIL import Image, ImageDraw
from gsplat.rendering import rasterization
from datasets.colmap import Parser

data_dir, factor, out = sys.argv[1], int(sys.argv[2]), sys.argv[3]
runs = [a.split("=", 1) for a in sys.argv[4:]]
YAWS = [0, 45, 70]                 # 정면(학습 각도), 45도, 70도
N_CAMS = 4

p = Parser(data_dir=data_dir, factor=factor, normalize=False, test_every=0)
K = torch.from_numpy(list(p.Ks_dict.values())[0]).float().cuda()
W, H = list(p.imsize_dict.values())[0]
idx = np.linspace(0, len(p.camtoworlds) - 1, N_CAMS + 2).astype(int)[1:-1]

def yaw(c2w, deg):
    a = np.deg2rad(deg); c, s = np.cos(a), np.sin(a)
    r = c2w.copy(); r[:3, :3] = c2w[:3, :3] @ np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return r

views = [yaw(p.camtoworlds[i], d) for i in idx for d in YAWS]
cols = []
for name, path in runs:
    s = torch.load(path, map_location="cuda", weights_only=True)["splats"]
    colors = torch.cat([s["sh0"], s["shN"]], 1)
    shd = int(np.sqrt(colors.shape[1])) - 1
    tiles = []
    for v in views:
        vm = torch.linalg.inv(torch.from_numpy(v).float().cuda())[None]
        with torch.no_grad():
            img, alpha, _ = rasterization(s["means"], s["quats"], torch.exp(s["scales"]),
                                          torch.sigmoid(s["opacities"]), colors, vm, K[None],
                                          W, H, sh_degree=shd)
        # 비어 있는 곳을 분홍으로 (Unity 분홍 배경 테스트와 같은 의미)
        pink = torch.tensor([1.0, 0.0, 1.0], device="cuda")
        rgb = img[0].clamp(0, 1) + pink * (1 - alpha[0])
        tiles.append(Image.fromarray((rgb.clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)))
    cols.append((name, tiles))
    del s; torch.cuda.empty_cache()

tw, th = W // 2, H // 2
canvas = Image.new("RGB", (tw * len(cols), (th) * len(views) + 40), "white")
d = ImageDraw.Draw(canvas)
for c, (name, tiles) in enumerate(cols):
    d.text((c * tw + 10, 10), name, fill="black")
    for r, t in enumerate(tiles):
        canvas.paste(t.resize((tw, th)), (c * tw, 40 + r * th))
canvas.save(out)
print("saved", out, "| rows = 카메라 x [정면, 45도, 70도], 분홍 = 비어 있는 곳")

In [ ]:
!python /content/nc/nc_patch.py {GS}/simple_trainer_difix3d.py
!cp /content/nc/nc_compare.py {GS}/

## 4. 데이터 준비 (드라이브 → 줄인 사진)
학습은 `images_4`(줄인 사진)만 읽음. `images` 는 이름 대조용이라 빈 파일로 둠 (4K 원본 복사 생략).

In [ ]:
import os, shutil, glob
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

mark = f"{DATA}/.ready_{FACTOR}"
if os.path.exists(mark):
    print("이미 준비됨 (건너뜀)")
else:
    shutil.rmtree(DATA, ignore_errors=True)
    os.makedirs(f"{DATA}/images"); os.makedirs(f"{DATA}/images_{FACTOR}")
    shutil.copytree(f"{DRIVE_DATA}/sparse", f"{DATA}/sparse")
    if FILL_POINTS:
        shutil.copy(FILL_POINTS, f"{DATA}/sparse/0/points3D.bin"); print("보강 시작점 사용:", FILL_POINTS)
    src = sorted(glob.glob(f"{DRIVE_DATA}/images/*"))

    def shrink(p):
        name = os.path.basename(p)
        open(f"{DATA}/images/{name}", "w").close()                    # 이름만 (빈 파일)
        im = Image.open(p).convert("RGB"); w, h = im.size
        im.resize((w // FACTOR, h // FACTOR), Image.LANCZOS).save(f"{DATA}/images_{FACTOR}/{name}", quality=95)
        return w, h

    print(len(src), "장 축소 중 (드라이브에서 읽어서 수 분 걸림)")
    with ThreadPoolExecutor(8) as ex:
        sizes = list(ex.map(shrink, src))
    print("원본", sizes[0], "->", Image.open(glob.glob(f"{DATA}/images_{FACTOR}/*")[0]).size)
    open(mark, "w").close()
print("sparse:", os.listdir(f"{DATA}/sparse"), "/ 사진:", len(os.listdir(f"{DATA}/images_{FACTOR}")), "장")

## 5. 학습 함수 (이어하기 · 정리)

In [ ]:
import os, glob, re

def ckpts(run):
    fs = glob.glob(f"{OUT}/{run}/ckpts/ckpt_*_rank0.pt")
    return sorted(fs, key=lambda p: int(re.findall(r"ckpt_(\d+)_rank0", p)[0]))

def final_ckpt(run, steps):
    return f"{OUT}/{run}/ckpts/ckpt_{steps - 1}_rank0.pt"

def train(run, steps, start=None, saves=(), fixes=(1_000_000,), env="", extra=""):
    """run 단계를 steps 까지 학습. 이미 끝났으면 건너뜀, 중간 저장본이 있으면 거기서 이어감."""
    final = final_ckpt(run, steps)
    if os.path.exists(final):
        print(f"[{run}] 이미 완료 (건너뜀)"); return final
    prev = ckpts(run)
    resume = prev[-1] if prev else start
    print(f"[{run}] 시작점:", resume or "처음부터")
    cmd = (f"{env} PYTHONPATH=/content/Difix3D python simple_trainer_difix3d.py mcmc "
           f"--data_dir {DATA} --data_factor {FACTOR} --result_dir {OUT}/{run} "
           f"--no-normalize-world-space --test_every 0 --sh_degree {SH} --strategy.cap-max {CAP} "
           f"--max_steps {steps} --eval_steps {steps} "
           f"--save_steps {' '.join(map(str, saves))} --fix_steps {' '.join(map(str, fixes))} "
           f"{'--ckpt ' + resume if resume else ''} {extra}")
    print(cmd)
    !cd {GS} && {cmd}
    if not os.path.exists(final):
        raise RuntimeError(f"[{run}] 끝까지 안 됨. 위 출력 마지막 부분 확인. 끊긴 거면 다시 '모두 실행' 하면 이어서 함")
    for p in ckpts(run):                      # 마지막 것만 남기고 중간 저장본 삭제 (드라이브 용량)
        if p != final: os.remove(p)
    return final
print("준비 OK")

## 6. ① 기본 학습 (보정 없음, 30,000 스텝)
비교 기준. 처음 실행할 때 gsplat CUDA 컴파일로 10~30분 멈춘 것처럼 보일 수 있음 (정상, "setting up cuda").

In [ ]:
BASE = train("base", 30_000, saves=range(5_000, 30_001, 5_000))

## 7. ② Difix 보정 학습 (①에 이어서 60,000 스텝까지)
4,000 스텝마다(8번): 카메라를 좌우로 돌린 각도에서 렌더링 → Difix 로 수정 → 학습 데이터에 추가.
중간에 끊겨서 이어갈 때는 보정 사진이 다시 15°부터 쌓임 (결과에 큰 지장 없음).

In [ ]:
dirs = 4 if PITCH_MAX > 0 else 2
stride = max(1, round(N_IMG * dirs / 120))   # 라운드당 약 120장 (방향 수에 맞춰 카메라를 솎음)
DIFIX = train("difix", 60_000, start=BASE,
              saves=range(35_000, 60_001, 5_000),
              fixes=range(32_000, 60_001, 4_000),
              env=(f"NC_STEP_DEG=15 NC_MAX_DEG=60 NC_STRIDE={stride} NC_PITCH_MAX={PITCH_MAX} NC_PITCH_STEP=10 "
                   f"NC_NO_ALPHA={1 if NO_ALPHA else 0}"))

## 8. ③ (선택) 배경 무작위화 — 해결 방향 3
1번 셀에서 `RUN_RBG = True` 일 때만 실행. ①과 같은 조건에 배경 무작위화만 추가.

In [ ]:
if RUN_RBG:
    train("rbg", 30_000, saves=range(5_000, 30_001, 5_000), extra="--random_bkgd")
else:
    print("건너뜀 (RUN_RBG = False)")

## 9. 비교 이미지 (같은 위치에서 정면 · 45° · 70°, 분홍 = 비어 있는 곳)

In [ ]:
import os
runs = {"base": final_ckpt("base", 30_000), "difix": final_ckpt("difix", 60_000), "rbg": final_ckpt("rbg", 30_000)}
args = " ".join(f"{k}={v}" for k, v in runs.items() if os.path.exists(v))
!cd {GS} && python nc_compare.py {DATA} {FACTOR} {OUT}/compare.png {args}
from IPython.display import Image as Show, display
if os.path.exists(f"{OUT}/compare.png"):
    display(Show(f"{OUT}/compare.png"))

## 10. PLY 내보내기 → SuperSplat / Unity
드라이브 `3DGS/colab/out/<SCENE>/ply` 에 저장.

In [ ]:
import os
os.makedirs(f"{OUT}/ply", exist_ok=True)
for k, v in runs.items():
    dst = f"{OUT}/ply/{SCENE}_{k}.ply"
    if os.path.exists(v) and not os.path.exists(dst):
        !python /content/nc/nc_export_ply.py {v} {dst}
!ls -la {OUT}/ply
print("\n끝. 런타임 > 런타임 연결 해제 및 삭제 로 GPU 를 반납할 것")